# Production LLM Serving on Cloud TPU v6e (Trillium) & v4 with vLLM

This notebook demonstrates how to deploy state-of-the-art open-weights models (such as **Google Gemma 3 27B**, **Gemma 2 9B**, or **Llama 3.1 8B**) on **Google Cloud TPU v6e (Trillium)** and **TPU v4** using the high-performance **vLLM** serving engine and **Google Cloud Vertex AI Endpoints**.

### Key Architectural Highlights
* **Hardware**: Google Cloud TPU v6e Trillium () with 4 TPU chips in a  mesh topology.
* **Inference Engine**: vLLM on TPU with PagedAttention and XLA kernel compilation.
* **Optimizations**: Prefix caching enabled for multi-turn chats, chunked prefill, and 16 GB shared memory allocation.
* **Orchestration**: Fully managed Vertex AI Endpoint with auto-scaling and dedicated SLAs.

## 1. Prerequisites and Environment Setup
Ensure you have enabled the required APIs and configured your Google Cloud project.

In [ ]:
# Install / Upgrade dependencies
!pip install --quiet --upgrade 'google-cloud-aiplatform>=1.64.0' 'pydantic>=2.5.0' rich tabulate

In [ ]:
import os
import getpass
from google.cloud import aiplatform

# Configuration variables - Replace with your project details or use environment variables
PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT', 'your-gcp-project-id')
REGION = os.environ.get('TPU_DEPLOYMENT_REGION', 'europe-west4')  # europe-west4 or us-central1 with TPU quota
BUCKET_URI = os.environ.get('BUCKET_URI', f'gs://{PROJECT_ID}-tpu-staging')

# Hugging Face token for gated models (e.g. Gemma 3, Llama 3.1)
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Enter your Hugging Face Read Token (optional): ')

print(f'Project ID: {PROJECT_ID}')
print(f'TPU Region: {REGION}')
print(f'Staging Bucket: {BUCKET_URI}')

# Initialize Vertex AI SDK
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=f'{BUCKET_URI}/temporal')

## 2. Configure Hardware and vLLM Serving Parameters
We configure the TPU accelerator type (), interconnect topology (), and the vLLM container runtime options.

In [ ]:
# Model and Hardware Specification
MODEL_ID = 'google/gemma-3-27b-it'  # Or 'google/gemma-2-9b-it', 'meta-llama/Llama-3.1-8B-Instruct'
PUBLISHER = 'google'
PUBLISHER_MODEL_ID = 'gemma-3-27b-it'

# Hardware topology
MACHINE_TYPE = 'ct6e-standard-4t'  # TPU v6e Trillium 4-chip
TPU_TOPOLOGY = '2x2'
TPU_COUNT = 4
TENSOR_PARALLEL_SIZE = 4
MAX_MODEL_LEN = 8192

# vLLM official Model Garden TPU serving container
VLLM_TPU_DOCKER_URI = 'us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20250819_0917_tpu_experimental_RC01'
USE_DEDICATED_ENDPOINT = True

## 3. Deployment Helper Function
This function uploads the model with custom vLLM entrypoint flags and provisions the Vertex AI dedicated endpoint.

In [ ]:
import datetime
from typing import Tuple

def deploy_vllm_tpu_endpoint(
    model_id: str,
    machine_type: str,
    tpu_topology: str,
    tensor_parallel_size: int,
    max_model_len: int = 8192,
    enable_prefix_caching: bool = True,
    enable_chunked_prefill: bool = False,
) -> Tuple[aiplatform.Model, aiplatform.Endpoint]:
    """Uploads and deploys an LLM to a Vertex AI TPU Endpoint with vLLM."""
    timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    display_name = f'{model_id.replace("/", "-")}-tpu-{timestamp}'

    # 1. Create Endpoint
    endpoint = aiplatform.Endpoint.create(
        display_name=f'{display_name}-endpoint',
        location=REGION,
        dedicated_endpoint_enabled=USE_DEDICATED_ENDPOINT,
    )

    # 2. Build Container Arguments
    container_args = [
        'python', '-m', 'vllm.entrypoints.api_server',
        '--host=0.0.0.0', '--port=7080',
        f'--model={model_id}',
        f'--tensor_parallel_size={tensor_parallel_size}',
        f'--max_model_len={max_model_len}',
        '--limit_mm_per_prompt.image=0',
    ]
    if enable_prefix_caching:
        container_args.append('--enable-prefix-caching')
    if enable_chunked_prefill:
        container_args.append('--enable-chunked-prefill')

    env_vars = {
        'MODEL_ID': model_id,
        'DEPLOY_SOURCE': 'notebook_deployment',
        'VLLM_USE_V1': '1',
    }
    if HF_TOKEN:
        env_vars['HF_TOKEN'] = HF_TOKEN

    # 3. Register Model in Vertex AI
    model = aiplatform.Model.upload(
        display_name=display_name,
        serving_container_image_uri=VLLM_TPU_DOCKER_URI,
        serving_container_args=container_args,
        serving_container_ports=[7080],
        serving_container_predict_route='/generate',
        serving_container_health_route='/ping',
        serving_container_environment_variables=env_vars,
        serving_container_shared_memory_size_mb=16384,  # 16 GB shared memory
        serving_container_deployment_timeout=4500,
        location=REGION,
    )

    # 4. Deploy to TPU Hardware
    model.deploy(
        endpoint=endpoint,
        machine_type=machine_type,
        tpu_topology=tpu_topology if 'x' in tpu_topology else None,
        deploy_request_timeout=1800,
        min_replica_count=1,
        max_replica_count=1,
    )
    return model, endpoint

## 4. Trigger Deployment
*Note: Cloud TPU provisioning and model weight loading into TPU HBM typically takes 15–25 minutes.*

In [ ]:
# Run deployment
# model, endpoint = deploy_vllm_tpu_endpoint(
#     model_id=MODEL_ID,
#     machine_type=MACHINE_TYPE,
#     tpu_topology=TPU_TOPOLOGY,
#     tensor_parallel_size=TENSOR_PARALLEL_SIZE,
#     max_model_len=MAX_MODEL_LEN,
# )
# print(f'Endpoint successfully deployed: {endpoint.resource_name}')

## 5. Inference Testing
Send text prompts directly to the vLLM TPU endpoint with customizable generation sampling parameters.

In [ ]:
import time

def query_tpu_endpoint(endpoint, prompt: str, max_tokens: int = 256, temperature: float = 0.7):
    instances = [{
        'prompt': prompt,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'raw_response': True,
    }]
    
    t0 = time.perf_counter()
    response = endpoint.predict(instances=instances, use_dedicated_endpoint=USE_DEDICATED_ENDPOINT)
    latency = time.perf_counter() - t0
    
    output_text = response.predictions[0] if response.predictions else ''
    print(f'=== Response (Latency: {latency*1000:.1f}ms) ===')
    print(output_text)
    return output_text

# Example invocation:
# query_tpu_endpoint(endpoint, 'Explain the role of Matrix Multiplication Units (MXUs) in Google TPUs.')

## 6. Teardown and Cleanup
To avoid continuous billing after testing is complete, always delete the endpoint and model.

In [ ]:
def cleanup_resources(endpoint, model):
    print('Undeploying and deleting endpoint...')
    endpoint.delete(force=True)
    print('Deleting model artifact...')
    model.delete()
    print('Cleanup complete.')

# cleanup_resources(endpoint, model)